#### Connecting to External Data Lake ( ADLS Gen 2)



######## Create a Service Principle in Azure and use it as a intermediatery between DB and ADLS Gen 2. Steps are outlined below.

https://learn.microsoft.com/en-us/azure/databricks/connect/storage/tutorial-azure-storage

#### Data Reading from Catelog (manually uploaded files)

###### 1. Reading a csv

In [0]:
df = spark.read.format("csv")\
                .option('inferSchema', 'true')\
                .option("header", "true")\
                .load("/Volumes/pyspark_catelog/pyspark_schema/pyspark_volume/BigMart Sales.csv")

df.display()

###### 2. Reading a JSON

In [0]:
df = spark.read.format("json")\
                .option('inferSchema', 'true')\
                .option("header", "true")\
                .load("/Volumes/pyspark_catelog/pyspark_schema/pyspark_volume/drivers.json")

df.display()

##### 3. Define and Update Schema

In [0]:
df.printSchema()    # prints a default schema of the df.

###### Create New Schema Definition and Use new schema to change the df.


In [0]:
# Step 1: Define New Schema (notice that original forename field is changed to firstname in the new schema.)
my_ddl_schema = '''
                code string,
                dob string,
                driverId long,
                driverRef string,
                name struct<firstname:string, surname:string>,
                nationality string,
                number string,
                url string
                 '''

# Step 2: read data using this new Schema
df = spark.read.format("json")\
                .schema(my_ddl_schema)\
                .load("/Volumes/pyspark_catelog/pyspark_schema/pyspark_volume/drivers.json")
    

df.printSchema()

### Creating a New DF

In [0]:
data1 = [
        ('1', 'sid'),
        ('2','matt')
        ]
schema1 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1, schema1)
df1.display()

### Writing Data into files in Storage

##### 1. To CSV  (currently inside Catelog> Volume)

In [0]:
from pyspark.sql.functions import col, to_json

newdata = [
        ('1', 'sid'),
        ('2','matt'),
        ('3','matty')
        ]
newschema = 'id STRING, name STRING'

newdf = spark.createDataFrame(newdata, newschema)
newdf.write.format('csv')\
    .option("header", "true")\
    .save("/Volumes/pyspark_catelog/pyspark_schema/pyspark_volume/written_csv")

##### 2. Append

In [0]:
newdf.write.format('csv')\
        .mode('append')\
        .save('/Volumes/pyspark_catelog/pyspark_schema/pyspark_volume/written_csv')

##### 3. Overwrite

In [0]:
newdf.write.format('csv')\
            .mode('overwrite')\
            .option('/Volumes/pyspark_catelog/pyspark_schema/pyspark_volume/written_csv')\
            .save()

##### 4. Error

In [0]:

newdf.write.format('csv')\
            .mode('error')\
            .option('/Volumes/pyspark_catelog/pyspark_schema/pyspark_volume/written_csv')\
            .save()

##### 5. Ignore

In [0]:
newdf.write.format('csv')\
            .mode('ignore')\
            .option('/Volumes/pyspark_catelog/pyspark_schema/pyspark_volume/written_csv')\
            .save()
     

##### 6. To Parquet File

In [0]:
newdf.write.format('parquet')\
            .mode('overwrite')\
            .option('/Volumes/pyspark_catelog/pyspark_schema/pyspark_volume/written_csv')\
            .save()

##### 7. To a Table

In [0]:
newdf.write.format('parquet')\
            .mode('overwrite')\
            .saveAsTable('my_table')

#### 8. To Delta Format (Delta lake - i.e. Parquet + Delta/ Transaction Logs)

In [0]:
###

###  Convert PySpark DF into Spark SQL.

##### Create a Temp View first (this will be destroyed once the session is terminated)
##### then use that Temp view for normal SQL operations

In [0]:
# Step 1: create a temp view from PySpark df
newdf.createTempView('my_view') 

In [0]:
%sql

select * from my_view

#### Convert spark SQL back into PySpark DF

In [0]:

df_sql = spark.sql("select * from my_view ")

df_sql.display()